Inspired by Bereta & Diamantis (2025), "The Shape of Consumer Behavior:
A Symbolic and Topological Analysis of Time Series"

Pipeline stages
----------------
1. Data loading - reads fixed stitched Google Trends CSVs from each term folder
2. KS test - Kolmogorov-Smirnov test for normality
3. Descriptive statistics - distributions, volatility, top average interest
4. STL + ADF - decomposition and stationarity checks
5. SAX / eSAX - symbolic time-series representations
6. TDA - persistence landscapes from sliding-window embeddings
7. Clustering - KMeans + Ward hierarchical clustering
8. Output - CSVs and plots for cluster review


In [1]:
from __future__ import annotations

import warnings
from pathlib import Path
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import LabelEncoder
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# CONFIG -- edit these for your environment
# ----------------------------------------------------------------------------

DATA_DIR = Path(r"C:\Python\CSUREMM\data_events")
OUTPUT_DIR = Path(r"C:\Python\CSUREMM\output")

# Stitched files live at: DATA_DIR/<term>/stitched/gt_fixed_<term>_<start>_<end>.csv
# (NOT directly inside the term folder, and the filename embeds the term name
# itself plus a date range that varies, so it can't be matched by a fixed
# literal filename -- it's located via the "stitched" subfolder + glob below.)
STITCHED_SUBDIR = "stitched"
STITCHED_GLOB = "gt_fixed_*.csv"

# --- SAX / eSAX parameter bounds ---
# These are no longer fixed analysis choices. They are upper/default bounds used
# by `infer_sax_params()` and can be tightened if you want faster runs.
MAX_WINDOW_SIZE = 365
MIN_WINDOW_SIZE = 90
MAX_N_SEGMENTS = 12
MIN_N_SEGMENTS = 6
ALPHABET_SIZE = 4
SAX_STEP = 7

# --- TDA parameter bounds ---
# EMBED_DIM and EMBED_TAU are now selected by `infer_tda_params()` from the
# normalized panel using autocorrelation and false-nearest-neighbor diagnostics.
MAX_EMBED_DIM = 8
MIN_EMBED_DIM = 2
MAX_EMBED_TAU = 60
MAX_HOMOLOGY_DIM = 1
N_LANDSCAPE_POINTS = 200
N_LANDSCAPE_LAYERS = 5

# --- Clustering ---
# k is now selected per representation by `select_k_by_diagnostics()`.
MIN_CLUSTERS = 2
MAX_CLUSTERS = 10
RANDOM_STATE = 42

# --- STL decomposition / ADF test (paper Section 3.2-3.4) ---
ROLLING_WINDOW = 91          # 13-week rolling avg (paper) -> 13*7 = 91 days
STL_PERIOD = 365              # seasonal period: paper used weekly seasonality
                               # implicitly tied to ~annual cycle; daily data ->
                               # 365-day period captures the same annual seasonality
ADF_ALPHA = 0.05               # significance level for ADF stationarity test

ALPHABET = list("abcdefghijklmnopqrstuvwxyz"[:ALPHABET_SIZE])


In [2]:
# ----------------------------------------------------------------------------
# 1. DATA LOADING
# ----------------------------------------------------------------------------

def load_all_series(
    data_dir: Path,
    stitched_subdir: str = STITCHED_SUBDIR,
    filename_glob: str = STITCHED_GLOB,
) -> dict[str, pd.Series]:
    """
    Walk every term subfolder in data_dir, read its stitched daily csv from
    DATA_DIR/<term>/<stitched_subdir>/<filename_glob>, and return
    {term_name: pandas Series indexed by date}.
    """
    series_dict: dict[str, pd.Series] = {}
    failed: list[tuple[str, str]] = []

    for folder in sorted(data_dir.iterdir()):
        if not folder.is_dir():
            continue

        stitched_dir = folder / stitched_subdir
        candidates = sorted(stitched_dir.glob(filename_glob))

        # Fallbacks make the notebook tolerant to older output names.
        if not candidates:
            candidates = sorted(stitched_dir.glob("gt_stitched_fixed_*.csv"))
        if not candidates:
            candidates = sorted(stitched_dir.glob("gt_stitched_*.csv"))

        if not candidates:
            failed.append((folder.name, f"missing file matching {filename_glob} in {stitched_dir}"))
            continue

        fpath = candidates[-1]

        try:
            df = pd.read_csv(fpath, parse_dates=["Time"])
            value_col = [c for c in df.columns if c != "Time"][0]
            ts = (
                df.set_index("Time")[value_col]
                .sort_index()
                .astype(float)
            )
            ts.name = folder.name
            series_dict[folder.name] = ts
        except Exception as e:
            failed.append((folder.name, str(e)))

    if failed:
        print(f"[load_all_series] {len(failed)} terms failed to load:")
        for name, err in failed:
            print(f"    {name}: {err}")
    print(f"[load_all_series] Loaded {len(series_dict)} term series.")
    return series_dict


def build_panel(series_dict: dict[str, pd.Series]) -> pd.DataFrame:
    """Align all series on their shared date index (outer join), as in paper Fig.5/Table 1."""
    panel = pd.DataFrame(series_dict)
    return panel


def summary_statistics(panel: pd.DataFrame) -> pd.DataFrame:
    """Replicates paper Table 1 (count, mean, std, min, 25/50/75pct, max) for every term."""
    desc = panel.describe(percentiles=[0.25, 0.5, 0.75]).T
    desc = desc.rename(columns={
        "count": "Count", "mean": "Mean", "std": "Std", "min": "Min",
        "25%": "25%", "50%": "50%", "75%": "75%", "max": "Max",
    })
    return desc[["Count", "Mean", "Std", "Min", "25%", "50%", "75%", "Max"]]


In [ ]:
# ----------------------------------------------------------------------------
# 2. GLOBAL PER-KEYWORD Z-NORMALIZATION
# ----------------------------------------------------------------------------

def zscore_series(s: pd.Series) -> pd.Series:
    """
    Global per-keyword Z-normalization.

    This removes stitched-level/range artifacts before KS, SAX/eSAX, and TDA.
    It is deliberately global across the full keyword series, not per-window.
    """
    x = s.astype(float)
    mu = x.mean(skipna=True)
    sigma = x.std(skipna=True, ddof=1)

    if pd.isna(sigma) or sigma == 0:
        return pd.Series(np.zeros(len(x)), index=x.index, name=x.name)

    return ((x - mu) / sigma).rename(x.name)


def zscore_panel(panel: pd.DataFrame) -> pd.DataFrame:
    """Column-wise global Z-normalization for an aligned keyword panel."""
    return panel.apply(zscore_series, axis=0)


def zscore_series_dict(series_dict: dict[str, pd.Series]) -> dict[str, pd.Series]:
    """Global Z-normalize each keyword series while preserving names/indexes."""
    return {
        term: zscore_series(s)
        for term, s in series_dict.items()
    }


In [18]:
# ----------------------------------------------------------------------------
# 3. SAX  (sliding-window Symbolic Aggregate approXimation)
# ----------------------------------------------------------------------------

def gaussian_breakpoints(alphabet_size: int) -> np.ndarray:
    """Gaussian breakpoints for equiprobable SAX bins."""
    return stats.norm.ppf(
        np.linspace(0, 1, alphabet_size + 1)[1:-1]
    )


def segment_bounds(window_size: int, n_segments: int) -> np.ndarray:
    """Integer PAA segment boundaries for a given window/segment choice."""
    return np.linspace(
        0,
        window_size,
        n_segments + 1,
        dtype=int,
    )


def znormalize(x: np.ndarray) -> np.ndarray:
    """Single-window Z-normalization, used only inside SAX/eSAX windows."""
    mu = np.nanmean(x)
    sigma = np.nanstd(x)

    if sigma == 0 or np.isnan(sigma):
        return np.zeros_like(x)

    return (x - mu) / sigma


def paa(
    x: np.ndarray,
    n_segments: int,
) -> np.ndarray:
    """Piecewise Aggregate Approximation with dynamic segment boundaries."""
    idx = segment_bounds(len(x), n_segments)

    return np.array([
        x[idx[i]:idx[i + 1]].mean()
        for i in range(n_segments)
    ])


def value_to_symbol(
    v: float,
    breakpoints: np.ndarray,
    alphabet: list[str] | None = None,
) -> str:
    if alphabet is None:
        alphabet = list("abcdefghijklmnopqrstuvwxyz"[:len(breakpoints) + 1])

    idx = np.searchsorted(
        breakpoints,
        v,
    )
    return alphabet[idx]


def infer_sax_params(
    panel_z: pd.DataFrame,
    min_window: int = MIN_WINDOW_SIZE,
    max_window: int = MAX_WINDOW_SIZE,
    min_segments: int = MIN_N_SEGMENTS,
    max_segments: int = MAX_N_SEGMENTS,
    alphabet_size: int = ALPHABET_SIZE,
) -> dict[str, int]:
    """
    Infer SAX/eSAX parameters from the available daily data.

    Heuristic:
    - window_size: longest stable seasonal window supported by the data, capped
      at one year and requiring at least two windows per series;
    - n_segments: about sqrt(window_size), bounded and never longer than window;
    - step: weekly stride for daily data to reduce redundant overlapping windows;
    - alphabet_size: kept at 4 unless you explicitly change it, because SAX
      Gaussian breakpoints are most stable with 4-6 bins.
    """
    valid_lengths = panel_z.notna().sum()
    min_len = int(valid_lengths.min())

    window_size = min(max_window, max(min_window, min_len // 2))
    window_size = max(2 * min_segments, window_size)

    n_segments = int(round(np.sqrt(window_size)))
    n_segments = min(max_segments, max(min_segments, n_segments))
    n_segments = min(n_segments, window_size // 2)

    step = 7 if window_size >= 90 else 1

    return {
        "window_size": int(window_size),
        "n_segments": int(n_segments),
        "alphabet_size": int(alphabet_size),
        "step": int(step),
    }


def sax_word(
    window: np.ndarray,
    n_segments: int,
    alphabet_size: int,
) -> str:
    """Compute one SAX word using dynamic breakpoints and PAA."""
    z = znormalize(window)
    seg_means = paa(z, n_segments)
    breakpoints = gaussian_breakpoints(alphabet_size)

    return "".join(
        value_to_symbol(v, breakpoints)
        for v in seg_means
    )


def build_sax_feature_matrix(
    series_dict: dict[str, pd.Series],
    window_size: int,
    n_segments: int,
    alphabet_size: int,
    step: int = SAX_STEP,
    spike: bool = True,  # added spike parameter to control inclusion of spike features
) -> tuple[pd.DataFrame, None]:

    feature_rows = {}
    breakpoints = gaussian_breakpoints(alphabet_size)

    for term, s in series_dict.items():
        x = s.dropna().values.astype(float)

        if len(x) < window_size:
            continue

        windows_raw = sliding_window_view(x, window_size)[::step]

        means = windows_raw.mean(axis=1, keepdims=True)
        stds = windows_raw.std(axis=1, keepdims=True)
        stds[stds == 0] = 1

        windows_z = (windows_raw - means) / stds

        numeric_words = []

        for w in windows_z:
            seg_means = paa(w, n_segments)
            numeric_words.append(
                np.searchsorted(
                    breakpoints,
                    seg_means,
                )
            )

        numeric_words = np.asarray(numeric_words, dtype=np.uint8)

        row = {
            f"sax_seg_{i + 1}": numeric_words[:, i].mean()
            for i in range(n_segments)
        }

        if spike:
            row.update({
                "max_z": np.nanmax(x),
                "min_z": np.nanmin(x),
                "range_z": np.nanmax(x) - np.nanmin(x),
                "p95_z": np.nanpercentile(x, 95),
                "p99_z": np.nanpercentile(x, 99),
                "spike_share_z2": np.mean(x > 2),
                "spike_share_z3": np.mean(x > 3),
            })

        feature_rows[term] = row

    feat_df = pd.DataFrame(feature_rows).T

    return feat_df, None




In [9]:
# ----------------------------------------------------------------------------
# 5. TDA  (sliding window embedding -> Vietoris-Rips -> persistence landscapes)
# ----------------------------------------------------------------------------

def autocorr_1d(x: np.ndarray, max_lag: int) -> np.ndarray:
    """Autocorrelation for lags 1..max_lag."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    x = x - x.mean()

    denom = np.dot(x, x)
    if denom == 0:
        return np.zeros(max_lag)

    return np.array([
        np.dot(x[:-lag], x[lag:]) / denom
        if lag < len(x) else np.nan
        for lag in range(1, max_lag + 1)
    ])


def choose_tau_from_autocorr(
    panel_z: pd.DataFrame,
    max_tau: int = MAX_EMBED_TAU,
) -> int:
    """
    Choose time delay using the first lag where median autocorrelation falls
    below 1/e. If none exists, use the first local minimum; otherwise fallback
    to lag 7 for daily data.
    """
    acfs = []

    for col in panel_z.columns:
        x = panel_z[col].dropna().values.astype(float)
        if len(x) > max_tau + 2:
            acfs.append(autocorr_1d(x, max_tau))

    if not acfs:
        return 7

    median_acf = np.nanmedian(np.vstack(acfs), axis=0)
    threshold_hits = np.where(median_acf < (1 / np.e))[0]

    if len(threshold_hits):
        return int(threshold_hits[0] + 1)

    for i in range(1, len(median_acf) - 1):
        if median_acf[i] < median_acf[i - 1] and median_acf[i] < median_acf[i + 1]:
            return int(i + 1)

    return 7


def false_nearest_fraction(
    x: np.ndarray,
    dim: int,
    tau: int,
    rtol: float = 10.0,
    atol: float = 2.0,
    max_points: int = 600,
) -> float:
    """
    Lightweight false-nearest-neighbor diagnostic for one series.

    Uses a random subsample to avoid a heavy O(n^2) calculation on long daily
    series. Lower values mean the embedding dimension is more adequate.
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    span_next = dim * tau
    n = len(x) - span_next
    if n <= dim + 2:
        return np.nan

    emb_d = np.column_stack([
        x[i * tau:i * tau + n]
        for i in range(dim)
    ])
    next_coord = x[dim * tau:dim * tau + n]

    if n > max_points:
        rng = np.random.default_rng(RANDOM_STATE)
        idx = np.sort(rng.choice(n, size=max_points, replace=False))
        emb_d = emb_d[idx]
        next_coord = next_coord[idx]
        n = max_points

    dists = np.linalg.norm(
        emb_d[:, None, :] - emb_d[None, :, :],
        axis=2,
    )
    np.fill_diagonal(dists, np.inf)

    nn_idx = np.argmin(dists, axis=1)
    dist_d = dists[np.arange(n), nn_idx]
    dist_next = np.sqrt(dist_d ** 2 + (next_coord - next_coord[nn_idx]) ** 2)

    attractor_scale = np.std(x)
    if attractor_scale == 0 or np.isnan(attractor_scale):
        return np.nan

    false = (
        ((dist_next - dist_d) / np.maximum(dist_d, 1e-12) > rtol)
        | (np.abs(next_coord - next_coord[nn_idx]) / attractor_scale > atol)
    )

    return float(np.mean(false))


def choose_embedding_dim(
    panel_z: pd.DataFrame,
    tau: int,
    min_dim: int = MIN_EMBED_DIM,
    max_dim: int = MAX_EMBED_DIM,
) -> int:
    """
    Choose embedding dimension using median false-nearest-neighbor fraction.
    Picks the first dimension with <10% false neighbors; otherwise the minimum
    of the diagnostic curve.
    """
    rows = {}

    for dim in range(min_dim, max_dim + 1):
        vals = []

        for col in panel_z.columns:
            x = panel_z[col].dropna().values.astype(float)
            val = false_nearest_fraction(x, dim, tau)
            if np.isfinite(val):
                vals.append(val)

        rows[dim] = np.nanmedian(vals) if vals else np.nan

    diag = pd.Series(rows, name="false_nearest_fraction")

    good = diag[diag < 0.10]
    if len(good):
        return int(good.index[0])

    return int(diag.idxmin()) if diag.notna().any() else 6


def infer_tda_params(panel_z: pd.DataFrame) -> tuple[dict[str, int], pd.DataFrame]:
    """Infer TDA embedding parameters and return diagnostics."""
    tau = choose_tau_from_autocorr(panel_z)
    dim = choose_embedding_dim(panel_z, tau)

    diagnostics = pd.DataFrame({
        "parameter": ["embed_tau", "embed_dim"],
        "selected_value": [tau, dim],
        "selection_rule": [
            "first median autocorrelation lag below 1/e, else first local minimum",
            "first median false-nearest-neighbor fraction below 10%, else minimum",
        ],
    })

    params = {
        "embed_dim": int(dim),
        "embed_tau": int(tau),
        "max_dim": int(MAX_HOMOLOGY_DIM),
        "n_landscape_points": int(N_LANDSCAPE_POINTS),
        "n_layers": int(N_LANDSCAPE_LAYERS),
    }

    return params, diagnostics


def sliding_window_embedding(x: np.ndarray, dim: int, tau: int) -> np.ndarray:
    """
    Takens-style sliding window embedding.
    Returns a point cloud of shape (n_points, dim).
    """
    n = len(x)
    span = (dim - 1) * tau
    n_points = n - span

    if n_points <= 0:
        raise ValueError("Series too short for the given embedding dimension/tau.")

    cloud = np.zeros((n_points, dim))

    for i in range(dim):
        cloud[:, i] = x[i * tau: i * tau + n_points]

    return cloud


def compute_persistence_diagrams(point_cloud: np.ndarray, max_dim: int = 1):
    """Uses ripser to compute persistence diagrams up to max_dim."""
    from ripser import ripser

    result = ripser(point_cloud, maxdim=max_dim)
    return result["dgms"]


def tent_function(t: np.ndarray, birth: float, death: float) -> np.ndarray:
    """Triangle/tent function for one persistence interval, evaluated at points t."""
    mid = (birth + death) / 2.0
    height = (death - birth) / 2.0

    if height <= 0:
        return np.zeros_like(t)

    val = height - np.abs(t - mid)
    return np.clip(val, 0, None)


def persistence_landscape(
    diagram: np.ndarray,
    n_points: int = N_LANDSCAPE_POINTS,
    n_layers: int = N_LANDSCAPE_LAYERS,
    t_range: tuple[float, float] | None = None,
) -> np.ndarray:
    """
    Convert a single persistence diagram into a stacked persistence landscape.
    """
    diagram = diagram[np.isfinite(diagram).all(axis=1)] if len(diagram) else diagram

    if len(diagram) == 0:
        return np.zeros((n_layers, n_points))

    if t_range is None:
        t_min, t_max = diagram[:, 0].min(), diagram[:, 1].max()
    else:
        t_min, t_max = t_range

    if t_max <= t_min:
        t_max = t_min + 1.0

    t = np.linspace(t_min, t_max, n_points)
    tents = np.array([
        tent_function(t, b, d)
        for b, d in diagram
    ])

    if tents.shape[0] == 0:
        return np.zeros((n_layers, n_points))

    sorted_tents = -np.sort(-tents, axis=0)
    n_available = sorted_tents.shape[0]

    landscape = np.zeros((n_layers, n_points))
    k = min(n_layers, n_available)
    landscape[:k, :] = sorted_tents[:k, :]

    return landscape


def landscape_to_vector(landscape: np.ndarray) -> np.ndarray:
    """Flatten a landscape into a single feature vector."""
    return landscape.flatten()


def build_tda_features(
    series_dict: dict[str, pd.Series],
    embed_dim: int,
    embed_tau: int,
    max_dim: int = MAX_HOMOLOGY_DIM,
    n_landscape_points: int = N_LANDSCAPE_POINTS,
    n_layers: int = N_LANDSCAPE_LAYERS,
):
    """
    Build TDA features from globally Z-normalized keyword series.

    This is the key fix for stitching artifacts: the point cloud geometry is
    constructed from normalized shapes, not raw stitched levels.
    """
    h1_rows, h01_rows, diagrams_raw = {}, {}, {}

    for term, s in series_dict.items():
        x = s.dropna().values.astype(float)

        try:
            cloud = sliding_window_embedding(x, embed_dim, embed_tau)
            dgms = compute_persistence_diagrams(cloud, max_dim=max_dim)
        except Exception as e:
            print(f"[TDA] {term} failed: {e}")
            continue

        diagrams_raw[term] = dgms

        h1_diagram = dgms[1] if len(dgms) > 1 else np.empty((0, 2))
        h1_landscape = persistence_landscape(
            h1_diagram,
            n_landscape_points,
            n_layers,
        )
        h1_rows[term] = landscape_to_vector(h1_landscape)

        h0_diagram = dgms[0] if len(dgms) > 0 else np.empty((0, 2))
        combined = (
            np.vstack([h0_diagram, h1_diagram])
            if len(h0_diagram) or len(h1_diagram)
            else np.empty((0, 2))
        )

        h01_landscape = persistence_landscape(
            combined,
            n_landscape_points,
            n_layers,
        )
        h01_rows[term] = landscape_to_vector(h01_landscape)

    h1_features = pd.DataFrame(h1_rows).T
    h01_features = pd.DataFrame(h01_rows).T

    return h1_features, h01_features, diagrams_raw


In [10]:
# ----------------------------------------------------------------------------
# 6. CLUSTERING  (KMeans + Hierarchical/Ward, with data-driven k)
# ----------------------------------------------------------------------------

@dataclass
class ClusterResult:
    method: str
    representation: str
    labels: pd.Series = field(default_factory=pd.Series)
    silhouette: float = np.nan
    davies_bouldin: float = np.nan
    n_clusters: int = np.nan


def valid_k_range(
    n_samples: int,
    min_k: int = MIN_CLUSTERS,
    max_k: int = MAX_CLUSTERS,
) -> range:
    """Valid k values for silhouette/KMeans."""
    upper = min(max_k, n_samples - 1)
    lower = min(min_k, upper)

    return range(lower, upper + 1)


def elbow_inertias(
    features: pd.DataFrame,
    k_range: range | None = None,
    random_state: int = RANDOM_STATE,
) -> pd.Series:
    if k_range is None:
        k_range = valid_k_range(len(features))

    inertias = {}

    for k in k_range:
        km = KMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=10,
        ).fit(features.values)
        inertias[k] = km.inertia_

    return pd.Series(inertias)


def detect_elbow_k(inertias: pd.Series) -> int:
    """
    Find elbow by maximum distance from each point to the line connecting the
    first and last inertia point. This is a no-dependency knee heuristic.
    """
    if len(inertias) <= 2:
        return int(inertias.index[-1])

    x = inertias.index.values.astype(float)
    y = inertias.values.astype(float)

    p1 = np.array([x[0], y[0]])
    p2 = np.array([x[-1], y[-1]])

    line = p2 - p1
    denom = np.linalg.norm(line)

    if denom == 0:
        return int(inertias.index[0])

    points = np.column_stack([x, y])
    distances = np.abs(np.cross(line, p1 - points)) / denom

    return int(x[np.argmax(distances)])


def cluster_diagnostics(
    features: pd.DataFrame,
    k_range: range | None = None,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """Compute inertia, silhouette, and DBI across candidate k values."""
    if k_range is None:
        k_range = valid_k_range(len(features))

    rows = []
    X = features.values

    for k in k_range:
        km = KMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=10,
        )
        labels = km.fit_predict(X)

        rows.append({
            "k": k,
            "inertia": km.inertia_,
            "silhouette": silhouette_score(X, labels),
            "davies_bouldin": davies_bouldin_score(X, labels),
        })

    diag = pd.DataFrame(rows).set_index("k")
    elbow_k = detect_elbow_k(diag["inertia"])
    diag["is_elbow_k"] = diag.index == elbow_k

    return diag


def select_k_by_diagnostics(
    features: pd.DataFrame,
    prefer: str = "silhouette",
    random_state: int = RANDOM_STATE,
) -> tuple[int, pd.DataFrame]:
    """
    Select k from diagnostics instead of presetting k=6.

    Default rule:
    - choose the highest silhouette score;
    - keep the elbow-k in the diagnostic table so event-spike clusters can be
      reviewed rather than forced into or out of k=6.
    """
    diag = cluster_diagnostics(
        features,
        random_state=random_state,
    )

    if prefer == "elbow":
        selected_k = int(diag.index[diag["is_elbow_k"]][0])
    elif prefer == "dbi":
        selected_k = int(diag["davies_bouldin"].idxmin())
    else:
        selected_k = int(diag["silhouette"].idxmax())

    diag["selected_k"] = diag.index == selected_k

    return selected_k, diag


def run_kmeans(
    features: pd.DataFrame,
    n_clusters: int | None = None,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.Series, float, float, int, pd.DataFrame]:
    X = features.values

    if n_clusters is None:
        n_clusters, diag = select_k_by_diagnostics(
            features,
            random_state=random_state,
        )
    else:
        diag = cluster_diagnostics(
            features,
            random_state=random_state,
        )
        diag["selected_k"] = diag.index == n_clusters

    km = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=10,
    )
    labels = km.fit_predict(X)

    sil = silhouette_score(X, labels) if len(set(labels)) > 1 else np.nan
    dbi = davies_bouldin_score(X, labels) if len(set(labels)) > 1 else np.nan

    return (
        pd.Series(labels, index=features.index, name="cluster"),
        sil,
        dbi,
        int(n_clusters),
        diag,
    )


def run_hierarchical(
    features: pd.DataFrame,
    n_clusters: int | None = None,
    method: str = "ward",
) -> tuple[pd.Series, float, float, np.ndarray, int]:
    X = features.values
    Z = linkage(X, method=method)

    if n_clusters is None:
        n_clusters, _ = select_k_by_diagnostics(features)

    labels = fcluster(
        Z,
        t=n_clusters,
        criterion="maxclust",
    ) - 1

    sil = silhouette_score(X, labels) if len(set(labels)) > 1 else np.nan
    dbi = davies_bouldin_score(X, labels) if len(set(labels)) > 1 else np.nan

    return (
        pd.Series(labels, index=features.index, name="cluster"),
        sil,
        dbi,
        Z,
        int(n_clusters),
    )


In [11]:
# ----------------------------------------------------------------------------
# 7. PLOTTING HELPERS
# ----------------------------------------------------------------------------

def plot_elbow(inertias: pd.Series, title: str, outpath: Path):
    plt.figure(figsize=(6, 4))
    plt.plot(inertias.index, inertias.values, marker="o")
    plt.xlabel("k (number of clusters)")
    plt.ylabel("Inertia")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()


def plot_clusters_by_series(
    series_dict: dict[str, pd.Series],
    labels: pd.Series,
    title: str,
    outpath: Path,
    n_clusters: int | None = None,
):
    if n_clusters is None:
        n_clusters = int(labels.max()) + 1

    fig, axes = plt.subplots(n_clusters, 1, figsize=(10, 2.2 * n_clusters), sharex=False)
    if n_clusters == 1:
        axes = [axes]
    for c in range(n_clusters):
        ax = axes[c]
        members = labels[labels == c].index
        for m in members:
            if m in series_dict:
                ax.plot(series_dict[m].index, series_dict[m].values, label=m, linewidth=0.9)
        ax.set_title(f"Cluster {c + 1} ({len(members)} terms)", fontsize=9)
        ax.legend(fontsize=6, loc="upper left", ncol=4)
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()


def plot_dendrogram(Z: np.ndarray, labels: list[str], title: str, outpath: Path):
    plt.figure(figsize=(14, 6))
    dendrogram(Z, labels=labels, leaf_rotation=90, leaf_font_size=6)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()


def plot_ks_bar(ks_df: pd.DataFrame, outpath: Path):
    df = ks_df.sort_values("KS Statistic")
    colors = ["tab:red" if r == "Yes" else "tab:green" for r in df["Reject H0 at 5%"]]
    plt.figure(figsize=(10, max(6, 0.18 * len(df))))
    plt.barh(df["Keyword"], df["KS Statistic"], color=colors)
    plt.xlabel("KS Statistic")
    plt.title("KS Test for Normality per Keyword (red = reject H0 at 5%)")
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()


In [ ]:
# ----------------------------------------------------------------------------
# MAIN PIPELINE
# ----------------------------------------------------------------------------

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # ---- 1. Load data ----
    print("=" * 70)
    print("Loading data")
    print("=" * 70)

    series_dict_raw = load_all_series(
        DATA_DIR,
        STITCHED_SUBDIR,
        STITCHED_GLOB,
    )
    panel_raw = build_panel(series_dict_raw)
    summary_statistics(panel_raw).to_csv(
        OUTPUT_DIR / "table1_summary_statistics_raw.csv"
    )

    # Global normalization used by KS, SAX/eSAX, and TDA.
    panel_z = zscore_panel(panel_raw)
    series_dict_z = {
        term: panel_z[term].dropna().rename(term)
        for term in panel_z.columns
    }
    summary_statistics(panel_z).to_csv(
        OUTPUT_DIR / "table1_summary_statistics_global_z.csv"
    )

    # Data-driven parameter selection.
    sax_params = infer_sax_params(panel_z)
    tda_params, tda_param_diag = infer_tda_params(panel_z)

    pd.DataFrame([sax_params]).to_csv(
        OUTPUT_DIR / "selected_sax_esax_parameters.csv",
        index=False,
    )
    tda_param_diag.to_csv(
        OUTPUT_DIR / "selected_tda_parameters.csv",
        index=False,
    )

    print("Selected SAX/eSAX parameters:")
    print(pd.Series(sax_params).to_string())
    print("\nSelected TDA parameters:")
    print(pd.Series(tda_params).to_string())
    
    # ---- 3. SAX ----
    print("\n" + "=" * 70)
    print("SAX on global-Z series")
    print("=" * 70)

    sax_features, sax_words = build_sax_feature_matrix(
        series_dict_z,
        sax_params["window_size"],
        sax_params["n_segments"],
        sax_params["alphabet_size"],
        sax_params["step"],
        spike = True
    )
    sax_features.to_csv(
        OUTPUT_DIR / "sax_feature_matrix_global_z.csv"
    )
    print(f"SAX feature matrix shape: {sax_features.shape}")

    sax_km_labels, sax_km_sil, sax_km_dbi, sax_k, sax_diag = run_kmeans(
        sax_features
    )
    sax_diag.to_csv(
        OUTPUT_DIR / "sax_k_selection_diagnostics.csv"
    )
    plot_elbow(
        sax_diag["inertia"],
        "SAX Elbow Method",
        OUTPUT_DIR / "fig_sax_elbow.png",
    )

    sax_hc_labels, sax_hc_sil, sax_hc_dbi, sax_Z, sax_hc_k = run_hierarchical(
        sax_features,
        n_clusters=sax_k,
    )

    plot_clusters_by_series(
        series_dict_z,
        sax_km_labels,
        f"SAX K-Means Clustering (k={sax_k}, global Z)",
        OUTPUT_DIR / "fig_sax_kmeans_clusters_global_z.png",
        n_clusters=sax_k,
    )
    plot_clusters_by_series(
        series_dict_z,
        sax_hc_labels,
        f"SAX Hierarchical Clustering (k={sax_hc_k}, global Z)",
        OUTPUT_DIR / "fig_sax_hierarchical_clusters_global_z.png",
        n_clusters=sax_hc_k,
    )
    plot_dendrogram(
        sax_Z,
        list(sax_features.index),
        "SAX Dendrogram (Ward)",
        OUTPUT_DIR / "fig_sax_dendrogram.png",
    )

    print(f"SAX selected k={sax_k}")
    print(f"SAX KMeans   -> Silhouette={sax_km_sil:.3f}, DBI={sax_km_dbi:.3f}")
    print(f"SAX Hierarch -> Silhouette={sax_hc_sil:.3f}, DBI={sax_hc_dbi:.3f}")

    # ---- 5. TDA ----
    print("\n" + "=" * 70)
    print("TDA on global-Z series")
    print("=" * 70)

    tda_h1, tda_h01, tda_diagrams = build_tda_features(
        series_dict_z,
        tda_params["embed_dim"],
        tda_params["embed_tau"],
        tda_params["max_dim"],
        tda_params["n_landscape_points"],
        tda_params["n_layers"],
    )
    tda_h1.to_csv(
        OUTPUT_DIR / "tda_h1_feature_matrix_global_z.csv"
    )
    tda_h01.to_csv(
        OUTPUT_DIR / "tda_h0h1_feature_matrix_global_z.csv"
    )

    print(f"TDA H1 feature matrix shape: {tda_h1.shape}")
    print(f"TDA H0+H1 feature matrix shape: {tda_h01.shape}")

    tda_km_labels, tda_km_sil, tda_km_dbi, tda_k, tda_diag = run_kmeans(
        tda_h1
    )
    tda_diag.to_csv(
        OUTPUT_DIR / "tda_h1_k_selection_diagnostics.csv"
    )
    plot_elbow(
        tda_diag["inertia"],
        "TDA H1 Elbow Method",
        OUTPUT_DIR / "fig_tda_h1_elbow.png",
    )
    plot_clusters_by_series(
        series_dict_z,
        tda_km_labels,
        f"TDA K-Means Clustering (H1, k={tda_k}, global Z)",
        OUTPUT_DIR / "fig_tda_kmeans_clusters_global_z.png",
        n_clusters=tda_k,
    )

    print(f"TDA H1 selected k={tda_k}")
    print(f"TDA KMeans (H1) -> Silhouette={tda_km_sil:.3f}, DBI={tda_km_dbi:.3f}")

    tda_hc_labels, tda_hc_sil, tda_hc_dbi, tda_Z, tda_hc_k = run_hierarchical(
        tda_h01,
        n_clusters=tda_k,
    )
    plot_clusters_by_series(
        series_dict_z,
        tda_hc_labels,
        f"TDA Hierarchical Clustering (H0+H1, k={tda_hc_k}, global Z)",
        OUTPUT_DIR / "fig_tda_hierarchical_clusters_global_z.png",
        n_clusters=tda_hc_k,
    )
    plot_dendrogram(
        tda_Z,
        list(tda_h01.index),
        "TDA Dendrogram (Ward, H0+H1)",
        OUTPUT_DIR / "fig_tda_dendrogram.png",
    )

    print(f"TDA Hierarchical (H0+H1) -> Silhouette={tda_hc_sil:.3f}, DBI={tda_hc_dbi:.3f}")

    # ---- 6. Summary comparison table ----
    print("\n" + "=" * 70)
    print("STEP 6: Final comparison table")
    print("=" * 70)

    comparison = pd.DataFrame({
        "SAX": {
            "Selected k": sax_k,
            "Silhouette (KMeans)": sax_km_sil,
            "DBI (KMeans)": sax_km_dbi,
            "Silhouette (Hierarchical)": sax_hc_sil,
            "DBI (Hierarchical)": sax_hc_dbi,
        },
        "TDA": {
            "Selected k": tda_k,
            "Silhouette (KMeans)": tda_km_sil,
            "DBI (KMeans)": tda_km_dbi,
            "Silhouette (Hierarchical)": tda_hc_sil,
            "DBI (Hierarchical)": tda_hc_dbi,
        },
    })
    comparison.to_csv(
        OUTPUT_DIR / "table6_clustering_comparison_global_z_selected_k.csv"
    )
    print(comparison.to_string())

    # ---- 7. Save cluster assignments ----
    all_labels = pd.DataFrame({
        "SAX_KMeans": sax_km_labels,
        "SAX_Hierarchical": sax_hc_labels,
        "TDA_KMeans_H1": tda_km_labels,
        "TDA_Hierarchical_H0H1": tda_hc_labels,
    })
    all_labels.to_csv(
        OUTPUT_DIR / "cluster_assignment.csv"
    )

    print(f"\nAll outputs written to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()


STEP 1: Loading data
[load_all_series] Loaded 492 term series.
Selected SAX/eSAX parameters:
window_size      365
n_segments        12
alphabet_size      4
step               7

Selected TDA parameters:
embed_dim               4
embed_tau               3
max_dim                 1
n_landscape_points    200
n_layers                5

STEP 2: Kolmogorov-Smirnov normality test on global-Z series
                   Keyword  KS Statistic       p-value Reject H0 at 5%
     women‘s_march_madness      0.525852 8.937537e-194             Yes
 gymnastics_rings_olympics      0.525730 1.117134e-193             Yes
       men‘s_march_madness      0.524882 9.675206e-299             Yes
           amanda_aldridge      0.523681 6.280601e-305             Yes
     hurricane_ian_tracker      0.523384  0.000000e+00             Yes
artistic_swimming_olympics      0.522895  0.000000e+00             Yes
      yellowstone_flooding      0.521152  0.000000e+00             Yes
        oscar_winners_2023      0.519

NameError: name 'esax_k' is not defined

In [16]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler


def tune_sax_fast(
    series_dict: dict[str, pd.Series],
    window_size: int,
    n_segments: int,
    step: int,
    alphabet_grid=(3, 4, 5, 6),
    k_grid=range(2, 8),
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """
    Fast SAX alphabet/k tuning.

    Important:
    - Input should already be globally Z-normalized if you want clustering to
      focus on shape/spike behavior rather than raw Google Trends scale.
    - Window-level Z-normalization is retained inside the SAX calculation.
    - PAA values are cached once, then reused for every alphabet size.
    """
    rows = []
    paa_cache: dict[str, np.ndarray] = {}

    # ---- 1. Precompute window-normalized PAA values once ----
    for term, s in series_dict.items():
        x = s.dropna().values.astype(float)

        if len(x) < window_size:
            continue

        windows_raw = sliding_window_view(x, window_size)[::step]

        means = windows_raw.mean(axis=1, keepdims=True)
        stds = windows_raw.std(axis=1, keepdims=True)
        stds[stds == 0] = 1
        windows_z = (windows_raw - means) / stds

        paa_cache[term] = np.vstack([
            paa(w, n_segments)
            for w in windows_z
        ])

    if not paa_cache:
        raise ValueError(
            "No series are long enough for the selected SAX window_size. "
            "Reduce window_size or check the input series."
        )

    # ---- 2. Re-bin cached PAA values for each alphabet size ----
    for alphabet_size in alphabet_grid:
        breakpoints = gaussian_breakpoints(alphabet_size)
        feature_rows = {}

        for term, paa_vals in paa_cache.items():
            symbols = np.searchsorted(
                breakpoints,
                paa_vals,
            )

            # Fixed-length SAX profile: average symbol index per segment.
            feature_rows[term] = symbols.mean(axis=0)

        X = pd.DataFrame(feature_rows).T.dropna()

        if len(X) < 3:
            continue

        X_scaled = StandardScaler().fit_transform(X)

        # k must be at least 2 and strictly less than n_samples for silhouette.
        valid_ks = [
            k for k in k_grid
            if 2 <= k < len(X_scaled)
        ]

        for k in valid_ks:
            labels = MiniBatchKMeans(
                n_clusters=k,
                random_state=random_state,
                n_init=10,
                batch_size=min(128, len(X_scaled)),
            ).fit_predict(X_scaled)

            rows.append({
                "alphabet_size": alphabet_size,
                "k": k,
                "silhouette": silhouette_score(X_scaled, labels),
                "davies_bouldin": davies_bouldin_score(X_scaled, labels),
                "n_terms": len(X),
            })

    if not rows:
        raise ValueError(
            "No valid alphabet/k combinations were evaluated. "
            "Check alphabet_grid, k_grid, and series length."
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["silhouette", "davies_bouldin"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )


In [17]:
# ----------------------------------------------------------------------------
# Optional SAX alphabet tuning block
# ----------------------------------------------------------------------------
# This block is intentionally self-contained. It works whether or not you have
# already run main(), because variables created inside main() are local and are
# not available here afterward.

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

series_dict_raw = load_all_series(
    DATA_DIR,
    STITCHED_SUBDIR,
    STITCHED_GLOB,
)

panel_raw = build_panel(series_dict_raw)
panel_z = zscore_panel(panel_raw)

series_dict_z = {
    term: panel_z[term].dropna().rename(term)
    for term in panel_z.columns
}

sax_params = infer_sax_params(panel_z)

alphabet_results = tune_sax_fast(
    series_dict=series_dict_z,
    window_size=sax_params["window_size"],
    n_segments=sax_params["n_segments"],
    step=sax_params["step"],
    alphabet_grid=(3, 4, 5, 6),
    k_grid=range(2, 8),
)

alphabet_results.to_csv(
    OUTPUT_DIR / "sax_alphabet_k_tuning_results.csv",
    index=False,
)

best = alphabet_results.iloc[0]
best_alphabet_size = int(best["alphabet_size"])
best_k = int(best["k"])

print("Selected SAX tuning result")
print("-" * 40)
print(f"alphabet_size : {best_alphabet_size}")
print(f"k             : {best_k}")
print(f"silhouette    : {best['silhouette']:.4f}")
print(f"davies_bouldin: {best['davies_bouldin']:.4f}")
print(f"n_terms       : {int(best['n_terms'])}")

print("\nTop 20 alphabet/k combinations:")
print(alphabet_results.head(20).to_string(index=False))

# Optional: build the final SAX feature matrix using the tuned alphabet.
sax_params_tuned = sax_params.copy()
sax_params_tuned["alphabet_size"] = best_alphabet_size

sax_features_tuned, _ = build_sax_feature_matrix(
    series_dict=series_dict_z,
    window_size=sax_params_tuned["window_size"],
    n_segments=sax_params_tuned["n_segments"],
    alphabet_size=sax_params_tuned["alphabet_size"],
    step=sax_params_tuned["step"],
    spike=True,
)

sax_features_tuned.to_csv(
    OUTPUT_DIR / "sax_feature_matrix_global_z_tuned_alphabet.csv"
)

pd.DataFrame([sax_params_tuned]).to_csv(
    OUTPUT_DIR / "selected_sax_parameters_tuned_alphabet.csv",
    index=False,
)


[load_all_series] Loaded 492 term series.
Selected SAX tuning result
----------------------------------------
alphabet_size : 6
k             : 2
silhouette    : 0.3761
davies_bouldin: 1.0394
n_terms       : 492

Top 20 alphabet/k combinations:
 alphabet_size  k  silhouette  davies_bouldin  n_terms
             6  2    0.376097        1.039356      492
             3  4    0.342651        1.043127      492
             3  3    0.328051        1.090182      492
             4  5    0.310852        1.012996      492
             3  2    0.310074        1.306634      492
             3  6    0.302537        0.990306      492
             5  5    0.289537        1.038482      492
             4  4    0.287126        1.030220      492
             4  3    0.281668        1.095910      492
             5  4    0.279876        1.113446      492
             6  5    0.276867        1.073326      492
             5  2    0.272464        1.361872      492
             5  7    0.268554        1.1

TypeError: build_sax_feature_matrix() got an unexpected keyword argument 'spike'

In [19]:
sax_features_tuned, _ = build_sax_feature_matrix(
    series_dict=series_dict_z,
    window_size=sax_params_tuned["window_size"],
    n_segments=sax_params_tuned["n_segments"],
    alphabet_size=sax_params_tuned["alphabet_size"],
    step=sax_params_tuned["step"],
    spike=True,
)

sax_features_tuned.to_csv(
    OUTPUT_DIR / "sax_feature_matrix_global_z_tuned_alphabet.csv"
)

pd.DataFrame([sax_params_tuned]).to_csv(
    OUTPUT_DIR / "selected_sax_parameters_tuned_alphabet.csv",
    index=False,
)